In [1]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform
from pathlib import Path

def simpson_dissimilarity_pdist(X):
    X = X.astype(bool)
    
    def simpson_pairwise(u, v):
        a = np.sum(u & v)          # overlap
        b = np.sum(u & ~v)         # unique to u
        c = np.sum(~u & v)         # unique to v
        denom = a + min(b, c)
        return min(b, c) / denom if denom > 0 else 0

    # pdist computes upper triangular distances, then squareform converts to full matrix
    dist_array = pdist(X, metric=simpson_pairwise)
    return squareform(dist_array)

def calculate_beta_diversity_optimized(scenario, cell_size, min_species_per_grid):
    # Load species distribution matrix
    file_path = Path(f'output/scenario={scenario}/cell={cell_size//1000}km') / f'{scenario}_{cell_size//1000}km_grid_species_matrix.xlsx'
    df = pd.read_excel(file_path)
    
    # Filter grids with fewer than min_species_per_grid species
    species_counts_per_grid = df.drop(columns='grid_index').sum(axis=1)
    filtered_df = df[species_counts_per_grid >= min_species_per_grid].reset_index(drop=True)
    
    grid_names = filtered_df['grid_index'].astype(str).values
    species_matrix = filtered_df.drop(columns='grid_index').values
    print(f"[INFO] Number of grids included: {len(grid_names)}")
    
    # Compute Simpson dissimilarity
    simpson_matrix = simpson_dissimilarity_pdist(species_matrix)
    
    # Convert to DataFrame
    simpson_df = pd.DataFrame(simpson_matrix, index=grid_names, columns=grid_names).round(3)
    
    # Save to ExcelW
    output_path = Path(f"output/scenario={scenario}/cell={cell_size//1000}km/threshold={min_species_per_grid}")
    output_path.mkdir(parents=True, exist_ok=True)
    out_file = output_path / f'{scenario}_{cell_size//1000}km_{min_species_per_grid}_beta_diversity_matrix.xlsx'
    simpson_df.to_excel(out_file)
    print(f"[INFO] Beta diversity matrix saved: {out_file}")
    out_file = output_path / f'{scenario}_{cell_size//1000}km_{min_species_per_grid}_grid_species_matrix.xlsx'
    filtered_df.to_excel(out_file, index=False)
    print(f"[INFO] Grid species matrix saved: {out_file}")
 
# Example usage
calculate_beta_diversity_optimized(scenario='all', cell_size=40000, min_species_per_grid=5)
calculate_beta_diversity_optimized(scenario='all', cell_size=40000, min_species_per_grid=1)

[INFO] Number of grids included: 4605
[INFO] Beta diversity matrix saved: output\scenario=all\cell=40km\threshold=5\all_40km_5_beta_diversity_matrix.xlsx
[INFO] Grid species matrix saved: output\scenario=all\cell=40km\threshold=5\all_40km_5_grid_species_matrix.xlsx
[INFO] Number of grids included: 5869
[INFO] Beta diversity matrix saved: output\scenario=all\cell=40km\threshold=1\all_40km_1_beta_diversity_matrix.xlsx
[INFO] Grid species matrix saved: output\scenario=all\cell=40km\threshold=1\all_40km_1_grid_species_matrix.xlsx


In [2]:
# import geopandas as gpd
# import pandas as pd
# import matplotlib.pyplot as plt
# import numpy as np
# from pathlib import Path
# from matplotlib.colors import Normalize
# from matplotlib import colormaps  # 新版推荐使用

# def plot_turnover_on_map(scenario, cell_size, threshold, seed):
#     # --------------------------
#     # 1. Load beta diversity matrix
#     # --------------------------
#     beta_file = Path(f"output/scenario={scenario}/cell={cell_size//1000}km/threshold={threshold}/"
#                      f"{scenario}_{cell_size//1000}km_{threshold}_beta_diversity_matrix.xlsx")
#     beta_df = pd.read_excel(beta_file, index_col=0)
#     grid_names = beta_df.index.astype(str).tolist()

#     # --------------------------
#     # 2. Randomly select target grid
#     # --------------------------
#     rng = np.random.default_rng(seed)
#     target_grid = rng.choice(grid_names)
#     print(f"[INFO] Randomly selected target grid: {target_grid}")

#     # --------------------------
#     # 3. Load grid map
#     # --------------------------
#     grid_file = Path(f"output/scenario={scenario}/cell={cell_size//1000}km/"
#                      f"{scenario}_{cell_size//1000}km_grid_map.geojson")
#     grid_gdf = gpd.read_file(grid_file)
#     grid_gdf["grid_index"] = grid_gdf["grid_index"].astype(str)

#     # --------------------------
#     # 4. Merge beta diversity values
#     # --------------------------
#     turnover_series = beta_df.loc[target_grid]
#     grid_gdf = grid_gdf.merge(turnover_series.rename("turnover"),
#                               left_on="grid_index", right_index=True, how="left")

#     # --------------------------
#     # 5. Plot grid map with vertical colorbar
#     # --------------------------
#     fig, ax = plt.subplots(figsize=(8, 8))
#     # Draw empty grids
#     grid_gdf.boundary.plot(ax=ax, color="lightgray", linewidth=0.5, zorder=1)

#     # Prepare colormap (新版用 colormaps)
#     cmap = colormaps["RdYlGn_r"]
#     norm = Normalize(vmin=0, vmax=1)

#     # Plot colored grids
#     grid_gdf.dropna(subset=["turnover"]).plot(
#         ax=ax,
#         column="turnover",
#         cmap=cmap,
#         edgecolor="black",
#         linewidth=0.5,
#         zorder=2
#     )

#     # Highlight target grid with thicker blue boundary
#     target_geom = grid_gdf.loc[grid_gdf["grid_index"] == target_grid, "geometry"].values[0]
#     gpd.GeoSeries([target_geom]).plot(ax=ax, facecolor="blue", edgecolor="blue", linewidth=2.5, zorder=3)

#     ax.set_title(f"Species turnover relative to grid: {target_grid}", fontsize=12)
#     ax.set_xticks([])
#     ax.set_yticks([])
#     ax.set_aspect("equal")
#     plt.tight_layout()

#     # Create vertical colorbar with same height as axes
#     sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
#     sm.set_array([])
#     cbar = fig.colorbar(sm, ax=ax, orientation="vertical", fraction=0.025, pad=0.02)
#     cbar.set_label("Turnover (Simpson dissimilarity)", fontsize=10)

#     # --------------------------
#     # 6. Save figure
#     # --------------------------
#     output_path = Path(f"output/scenario={scenario}/cell={cell_size//1000}km/threshold={threshold}")
#     output_path.mkdir(parents=True, exist_ok=True)
#     out_file = output_path / f"{scenario}_{cell_size//1000}km_{threshold}_turnover_{seed}.tif"
#     plt.savefig(out_file, bbox_inches="tight", dpi=300)
#     plt.close()

#     print(f"[INFO] Turnover map saved: {out_file}")

# # # Example usage
# # for i in range(10):
# #     plot_turnover_on_map(scenario='all', cell_size=40000, threshold=1, seed=i)